# Recorte da cidade-alvo e exportação inicial

Este notebook prepara o primeiro recorte do POC a partir do ZIP original do snapshot mensal do CNPJ.

## Fluxo recomendado no Colab

1. Manter no Google Drive apenas o ZIP mensal original do snapshot.
2. Montar o Drive no Colab e copiar esse ZIP para `/content`.
3. Executar este notebook.
4. O notebook copia o ZIP para `original`, extrai o conteúdo para `extraido` e remove a camada temporária de ZIPs internos.
5. Em seguida, processa uma família por vez, salva partes em disco e produz o primeiro recorte em `processado/recorte`.

Configuração atual do POC:

- cidade-alvo: `Praia Grande - SP`
- código do município no layout do CNPJ: `6921`

In [ ]:
from pathlib import Path
import shutil
import zipfile
from google.colab import drive

try:
    import pandas as pd
    import pyarrow  # noqa: F401
except ImportError:
    !pip -q install pandas pyarrow
    import pandas as pd
    import pyarrow  # noqa: F401

try:
    import polars as pl
except ImportError:
    !pip -q install polars pyarrow
    import polars as pl

SNAPSHOT_MES = '2026-04'
NOME_ARQUIVO_ZIP = '2026-04.zip'
DRIVE_RAIZ = Path('/content/drive/MyDrive/geoBusiness/dados_brutos/cnpj')
CAMINHO_ZIP_DRIVE = DRIVE_RAIZ / SNAPSHOT_MES / 'original' / NOME_ARQUIVO_ZIP
ENTRADA_LOCAL = Path('/content/entrada_cnpj')
CAMINHO_ZIP_PRINCIPAL = ENTRADA_LOCAL / NOME_ARQUIVO_ZIP
RAIZ_DADOS = Path('/content/dados_brutos/cnpj')
CODIGO_MUNICIPIO = '6921'
TAMANHO_CHUNK_ESTABELECIMENTOS = 100_000
TAMANHO_CHUNK_CADASTRAIS = 200_000

drive.mount('/content/drive')
ENTRADA_LOCAL.mkdir(parents=True, exist_ok=True)
if not CAMINHO_ZIP_DRIVE.exists():
    raise FileNotFoundError(f'Arquivo não encontrado no Drive: {CAMINHO_ZIP_DRIVE}')

precisa_recopiar = False

if not CAMINHO_ZIP_PRINCIPAL.exists():
    precisa_recopiar = True
else:
    tamanho_drive = CAMINHO_ZIP_DRIVE.stat().st_size
    tamanho_local = CAMINHO_ZIP_PRINCIPAL.stat().st_size

    if tamanho_local != tamanho_drive:
        print('Arquivo local incompleto ou divergente. Será copiado novamente.')
        CAMINHO_ZIP_PRINCIPAL.unlink()
        precisa_recopiar = True
    elif not zipfile.is_zipfile(CAMINHO_ZIP_PRINCIPAL):
        print('Arquivo local inválido. Será copiado novamente.')
        CAMINHO_ZIP_PRINCIPAL.unlink()
        precisa_recopiar = True

if precisa_recopiar:
    shutil.copy2(CAMINHO_ZIP_DRIVE, CAMINHO_ZIP_PRINCIPAL)

print('ZIP no Drive:', CAMINHO_ZIP_DRIVE)
print('ZIP local:', CAMINHO_ZIP_PRINCIPAL)
print('Tamanho no Drive:', CAMINHO_ZIP_DRIVE.stat().st_size)
print('Tamanho local:', CAMINHO_ZIP_PRINCIPAL.stat().st_size)
print('ZIP local válido?', zipfile.is_zipfile(CAMINHO_ZIP_PRINCIPAL))
CAMINHO_ZIP_PRINCIPAL

In [ ]:
FAMILIAS = {
    'Empresas': 'empresas',
    'Estabelecimentos': 'estabelecimentos',
    'Socios': 'socios',
    'Simples': 'simples',
    'Cnaes': 'dominios',
    'Motivos': 'dominios',
    'Municipios': 'dominios',
    'Naturezas': 'dominios',
    'Paises': 'dominios',
    'Qualificacoes': 'dominios',
}

def classificar_zip(nome: str) -> str:
    for prefixo, familia in FAMILIAS.items():
        if nome.startswith(prefixo):
            return familia
    raise ValueError(f'ZIP interno não reconhecido: {nome}')

def preparar_snapshot(caminho_zip_principal: Path, raiz_dados: Path, snapshot_mes: str) -> Path:
    if not caminho_zip_principal.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {caminho_zip_principal}')

    snapshot = raiz_dados / snapshot_mes
    pacote_original = snapshot / 'original'
    extraido_texto = snapshot / 'extraido'
    processado = snapshot / 'processado'
    metadados = snapshot / 'metadados'
    temporario = snapshot / '_tmp_subarquivos_zip'

    for pasta in [pacote_original, extraido_texto, processado / 'recorte', processado / 'preparado', processado / 'analitico', metadados, temporario]:
        pasta.mkdir(parents=True, exist_ok=True)

    destino_zip = pacote_original / caminho_zip_principal.name
    if caminho_zip_principal.resolve() != destino_zip.resolve() and not destino_zip.exists():
        shutil.copy2(caminho_zip_principal, destino_zip)
    caminho_zip_principal = destino_zip

    if not any(extraido_texto.rglob('*')):
        with zipfile.ZipFile(caminho_zip_principal) as zip_principal:
            zip_principal.extractall(temporario)

        zips_internos = sorted(
            arquivo for arquivo in temporario.rglob('*.zip') if arquivo.is_file()
        )

        if not zips_internos:
            raise FileNotFoundError(
                f'Nenhum ZIP interno foi encontrado dentro de {caminho_zip_principal}'
            )

        for zip_interno in zips_internos:
            familia = classificar_zip(zip_interno.name)
            destino_familia = extraido_texto / familia
            destino_familia.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_interno) as zip_familia:
                zip_familia.extractall(destino_familia)

        shutil.rmtree(temporario)
    elif temporario.exists():
        shutil.rmtree(temporario)

    return snapshot

SNAPSHOT = preparar_snapshot(CAMINHO_ZIP_PRINCIPAL, RAIZ_DADOS, SNAPSHOT_MES)
SNAPSHOT

In [ ]:
colunas_empresas = [
    'cnpj_basico', 'razao_social', 'natureza_juridica', 'qualificacao_responsavel',
    'capital_social', 'porte_empresa', 'ente_federativo_responsavel'
]
colunas_estabelecimentos = [
    'cnpj_basico', 'cnpj_ordem', 'cnpj_dv', 'identificador_matriz_filial', 'nome_fantasia',
    'situacao_cadastral', 'data_situacao_cadastral', 'motivo_situacao_cadastral',
    'nome_cidade_exterior', 'pais', 'data_inicio_atividade', 'cnae_fiscal_principal',
    'cnaes_secundarios', 'tipo_logradouro', 'logradouro', 'numero', 'complemento', 'bairro',
    'cep', 'uf', 'municipio', 'ddd_1', 'telefone_1', 'ddd_2', 'telefone_2', 'ddd_fax', 'fax',
    'correio_eletronico', 'situacao_especial', 'data_situacao_especial'
]
colunas_simples = [
    'cnpj_basico', 'opcao_pelo_simples', 'data_opcao_pelo_simples', 'data_exclusao_do_simples',
    'opcao_pelo_mei', 'data_opcao_pelo_mei', 'data_exclusao_do_mei'
]

In [ ]:
def listar_arquivos(pasta: Path) -> list[Path]:
    return sorted([arquivo for arquivo in pasta.iterdir() if arquivo.is_file()])

def recriar_pasta(pasta: Path) -> None:
    if pasta.exists():
        shutil.rmtree(pasta)
    pasta.mkdir(parents=True, exist_ok=True)

def salvar_parte(df: pd.DataFrame, pasta_partes: Path, prefixo: str, indice: int) -> None:
    destino = pasta_partes / f'{prefixo}_{indice:05d}.parquet'
    df.to_parquet(destino, index=False)

def consolidar_partes(pasta_partes: Path, destino_final: Path) -> pl.DataFrame:
    arquivos = sorted(pasta_partes.glob('*.parquet'))
    if not arquivos:
        raise FileNotFoundError(f'Nenhuma parte parquet encontrada em {pasta_partes}')
    consolidado = pl.scan_parquet([str(arquivo) for arquivo in arquivos]).collect(streaming=True)
    consolidado.write_parquet(destino_final)
    shutil.rmtree(pasta_partes)
    return consolidado

def processar_estabelecimentos() -> tuple[pl.DataFrame, set[str]]:
    pasta_familia = SNAPSHOT / 'extraido' / 'estabelecimentos'
    pasta_partes = saida_recorte / '_partes_estabelecimentos'
    recriar_pasta(pasta_partes)

    total_linhas = 0
    total_filtradas = 0
    indice_parte = 0

    for arquivo in listar_arquivos(pasta_familia):
        for chunk in pd.read_csv(
            arquivo,
            sep=';',
            header=None,
            names=colunas_estabelecimentos,
            dtype=str,
            encoding='latin1',
            quotechar='"',
            keep_default_na=False,
            chunksize=TAMANHO_CHUNK_ESTABELECIMENTOS,
        ):
            total_linhas += len(chunk)
            filtrado = chunk.loc[chunk['municipio'] == CODIGO_MUNICIPIO].copy()
            if filtrado.empty:
                continue
            salvar_parte(filtrado, pasta_partes, 'estabelecimentos', indice_parte)
            total_filtradas += len(filtrado)
            indice_parte += 1

    if indice_parte == 0:
        raise ValueError(f'Nenhum estabelecimento encontrado para o município {CODIGO_MUNICIPIO}')

    recorte = consolidar_partes(pasta_partes, saida_recorte / 'estabelecimentos_municipio.parquet')
    cnpjs_basicos = set(recorte.select('cnpj_basico').unique().to_series().to_list())
    recorte.select('cnpj_basico').unique().write_parquet(saida_recorte / 'cnpjs_basicos_municipio.parquet')

    print(f'Estabelecimentos lidos: {total_linhas}')
    print(f'Estabelecimentos filtrados: {total_filtradas}')
    print(f'CNPJs básicos únicos no município: {len(cnpjs_basicos)}')

    return recorte, cnpjs_basicos

def processar_familia_por_cnpj(
    nome_familia: str,
    colunas: list[str],
    cnpjs_basicos: set[str],
    tamanho_chunk: int,
    nome_arquivo_saida: str,
) -> pl.DataFrame:
    pasta_familia = SNAPSHOT / 'extraido' / nome_familia
    pasta_partes = saida_recorte / f'_partes_{nome_familia}'
    recriar_pasta(pasta_partes)

    total_linhas = 0
    total_filtradas = 0
    indice_parte = 0

    for arquivo in listar_arquivos(pasta_familia):
        for chunk in pd.read_csv(
            arquivo,
            sep=';',
            header=None,
            names=colunas,
            dtype=str,
            encoding='latin1',
            quotechar='"',
            keep_default_na=False,
            chunksize=tamanho_chunk,
        ):
            total_linhas += len(chunk)
            filtrado = chunk.loc[chunk['cnpj_basico'].isin(cnpjs_basicos)].copy()
            if filtrado.empty:
                continue
            salvar_parte(filtrado, pasta_partes, nome_familia, indice_parte)
            total_filtradas += len(filtrado)
            indice_parte += 1

    if indice_parte == 0:
        vazio = pl.DataFrame({coluna: [] for coluna in colunas})
        vazio.write_parquet(saida_recorte / nome_arquivo_saida)
        shutil.rmtree(pasta_partes)
        print(f'Nenhum registro filtrado em {nome_familia}.')
        return vazio

    recorte = consolidar_partes(pasta_partes, saida_recorte / nome_arquivo_saida)
    print(f'{nome_familia.title()} lidos: {total_linhas}')
    print(f'{nome_familia.title()} filtrados: {total_filtradas}')
    return recorte


In [ ]:
saida_recorte = SNAPSHOT / 'processado' / 'recorte'
saida_recorte.mkdir(parents=True, exist_ok=True)

recorte_estabelecimentos, cnpjs_basicos_municipio = processar_estabelecimentos()
recorte_empresas = processar_familia_por_cnpj(
    nome_familia='empresas',
    colunas=colunas_empresas,
    cnpjs_basicos=cnpjs_basicos_municipio,
    tamanho_chunk=TAMANHO_CHUNK_CADASTRAIS,
    nome_arquivo_saida='empresas_municipio.parquet',
)
recorte_simples = processar_familia_por_cnpj(
    nome_familia='simples',
    colunas=colunas_simples,
    cnpjs_basicos=cnpjs_basicos_municipio,
    tamanho_chunk=TAMANHO_CHUNK_CADASTRAIS,
    nome_arquivo_saida='simples_municipio.parquet',
)

recorte_estabelecimentos.shape, recorte_empresas.shape, recorte_simples.shape

In [ ]:
sorted([arquivo.name for arquivo in saida_recorte.iterdir() if arquivo.is_file()])